# Clustering Gempa Bumi di Wilayah Indonesia
---
Studi kasus unsupervised learning: mengelompokkan kejadian gempa bumi di wilayah Indonesia berdasarkan **lokasi, kedalaman, dan magnitudo** menggunakan K-Means, Hierarchical Clustering, dan DBSCAN.

**Sumber data:** USGS Earthquake Catalog (United States Geological Survey), https://earthquake.usgs.gov/earthquakes/search/

### Daftar Isi:
1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modeling
    * K-Means
    * Hierarchical Clustering
    * DBSCAN
5. Evaluation
6. Interpretasi Cluster
7. Deployment

#### 1. Business Understanding

Indonesia berada di pertemuan tiga lempeng tektonik besar (Indo-Australia, Eurasia, dan Pasifik), sehingga menjadi salah satu negara dengan aktivitas gempa tertinggi di dunia. Gempa di Indonesia tidak tersebar acak: gempa cenderung terkonsentrasi di zona subduksi (barat Sumatra, selatan Jawa hingga Nusa Tenggara), zona tumbukan di Maluku dan Laut Banda, serta patahan aktif di Sulawesi dan Papua. Kedalamannya juga bervariasi, dari gempa dangkal (< 70 km) yang umumnya lebih merusak sampai gempa dalam (> 300 km) di bawah Laut Banda dan Laut Flores.

Dataset yang digunakan adalah katalog gempa dari USGS untuk wilayah Indonesia periode 2015–2024 dengan magnitudo ≥ 4,5. Setiap baris mewakili satu kejadian gempa, dengan informasi waktu, koordinat, kedalaman, magnitudo, kualitas pengukuran, dan deskripsi lokasi.

Permasalahan yang ingin diselesaikan adalah bagaimana menemukan kelompok gempa yang memiliki karakteristik serupa berdasarkan lokasi (`latitude`, `longitude`), kedalaman (`depth`), dan kekuatan (`mag`) tanpa menggunakan label apa pun.

Tujuan analisis adalah:

* Mengidentifikasi kelompok (zona) gempa berdasarkan lokasi, kedalaman, dan magnitudo.
* Membandingkan hasil clustering menggunakan algoritma K-Means, Hierarchical Clustering, dan DBSCAN.
* Menentukan algoritma clustering terbaik berdasarkan Silhouette Score dan Davies-Bouldin Index.
* Menginterpretasikan karakteristik tiap cluster (wilayah dominan, kedalaman rata-rata, magnitudo rata-rata) sebagai gambaran zona rawan gempa di Indonesia.

**Alasan pemilihan dataset untuk clustering:**
1. Data tidak memiliki label kelompok, sehingga sesuai dengan pendekatan unsupervised learning.
2. Gempa secara alami membentuk kelompok berdasarkan struktur tektonik, sehingga hasil cluster dapat divalidasi secara logis dengan pengetahuan geologi.
3. Terdapat beberapa fitur numerik yang relevan (koordinat, kedalaman, magnitudo) dengan skala yang berbeda-beda, sehingga tahap standardisasi menjadi penting.
4. Hasil clustering bermanfaat untuk pemetaan zona rawan gempa dan perencanaan mitigasi bencana.

**Mengambil data.** Data diunduh langsung dari API USGS (FDSN Event Web Service) dengan filter:
* Wilayah: latitude −11 s.d. 6, longitude 95 s.d. 141 (mencakup seluruh Indonesia)
* Periode: 1 Januari 2015 – 31 Desember 2024
* Magnitudo minimum: 4,5

Jika file `gempa_indonesia_usgs.csv` sudah ada di folder yang sama, file tersebut yang dibaca. Jika belum ada, data diunduh dari USGS lalu disimpan.

In [ ]:
# Import library pandas dan numpy
import pandas as pd
import numpy as np
import os

FILE = "gempa_indonesia_usgs.csv"

URL = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
    "&starttime=2015-01-01&endtime=2024-12-31"
    "&minlatitude=-11&maxlatitude=6"
    "&minlongitude=95&maxlongitude=141"
    "&minmagnitude=4.5&eventtype=earthquake&orderby=time-asc"
)

if os.path.exists(FILE):
    df = pd.read_csv(FILE)
else:
    df = pd.read_csv(URL)
    df.to_csv(FILE, index=False)

# Menampilkan 5 data pertama
df.head()

#### 2. Data Understanding

Tahap Data Understanding bertujuan untuk memahami struktur, tipe data, jumlah data, serta karakteristik awal dari dataset sebelum pemodelan.

In [ ]:
# Melihat ukuran dataset
df.shape

In [ ]:
# Melihat informasi dataset
df.info()

Penjelasan kolom utama:

| Kolom | Keterangan |
|---|---|
| `time` | Waktu kejadian gempa (UTC) |
| `latitude`, `longitude` | Koordinat episenter |
| `depth` | Kedalaman hiposenter (km) |
| `mag` | Magnitudo gempa |
| `magType` | Jenis skala magnitudo (mb, mww, dll.) |
| `nst`, `gap`, `dmin`, `rms` | Indikator kualitas pengukuran (jumlah stasiun, celah azimut, jarak stasiun terdekat, residu waktu tempuh) |
| `place` | Deskripsi lokasi dalam teks |
| `horizontalError`, `depthError`, `magError` | Estimasi galat pengukuran |

In [ ]:
# Statistik deskriptif variabel numerik
df.describe()

In [ ]:
# Statistik deskriptif variabel kategorikal
df.describe(include="object")

In [ ]:
# Mengecek missing value pada setiap kolom
df.isnull().sum()

Berdasarkan output di atas, dataset terdiri dari **8.731 baris dan 22 kolom**. Missing value hanya terdapat pada kolom `nst` (6.406 data), `magError` (111 data), dan `magNst` (111 data). Ketiga kolom tersebut merupakan indikator kualitas pengukuran dan **tidak digunakan** sebagai fitur clustering. Empat fitur yang digunakan (`latitude`, `longitude`, `depth`, `mag`) tidak memiliki missing value sama sekali.

#### Exploratory Data Analysis (EDA)

In [ ]:
# Import library visualisasi
import matplotlib.pyplot as plt
import seaborn as sns

Pertama, persebaran episenter gempa divisualisasikan menggunakan scatterplot `longitude` terhadap `latitude`. Warna titik menunjukkan kedalaman gempa.

In [ ]:
plt.figure(figsize=(14, 6))

plt.scatter(
    df["longitude"],
    df["latitude"],
    c=df["depth"],
    cmap="viridis_r",
    s=8,
    alpha=0.7
)

plt.colorbar(label="Kedalaman (km)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Persebaran Episenter Gempa M≥4,5 di Wilayah Indonesia (2015–2024)")
plt.show()

Scatterplot di atas membentuk pola yang menyerupai peta busur kepulauan Indonesia. Titik-titik gempa mengikuti jalur subduksi di sebelah barat Sumatra dan selatan Jawa–Nusa Tenggara, lalu memadat di kawasan Laut Banda, Maluku, Sulawesi Utara, dan Papua. Gempa dalam (warna gelap, > 300 km) terlihat terkonsentrasi di bawah Laut Jawa–Laut Flores–Laut Banda (latitude sekitar −6 s.d. −8) serta di Laut Sulawesi bagian utara (sekitar longitude 123–125, latitude 3–5).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.histplot(df["depth"], bins=50, ax=axes[0])
axes[0].set_title("Distribusi Kedalaman Gempa")
axes[0].set_xlabel("Kedalaman (km)")

sns.histplot(df["mag"], bins=30, ax=axes[1])
axes[1].set_title("Distribusi Magnitudo Gempa")
axes[1].set_xlabel("Magnitudo")

plt.tight_layout()
plt.show()

Distribusi kedalaman sangat condong ke kanan (right-skewed): median kedalaman hanya sekitar 41 km, tetapi nilai maksimumnya mencapai 673 km. Artinya, sebagian besar gempa merupakan gempa dangkal dan hanya sedikit yang berada di kedalaman ratusan kilometer. Distribusi magnitudo juga condong ke kanan (median 4,7, maksimum 7,6), sesuai hukum Gutenberg–Richter: gempa kecil jauh lebih sering terjadi dibandingkan gempa besar.

In [ ]:
# Hubungan antara longitude dan kedalaman (penampang barat–timur)
plt.figure(figsize=(14, 5))

plt.scatter(
    df["longitude"],
    df["depth"],
    s=8,
    alpha=0.5
)

plt.gca().invert_yaxis()
plt.xlabel("Longitude")
plt.ylabel("Kedalaman (km)")
plt.title("Penampang Kedalaman Gempa dari Barat ke Timur")
plt.show()

In [ ]:
# Korelasi antar fitur yang akan digunakan
plt.figure(figsize=(6, 5))

sns.heatmap(
    df[["latitude", "longitude", "depth", "mag"]].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Korelasi Antar Fitur")
plt.show()

Korelasi antar fitur sangat rendah (seluruh nilai korelasi berada di antara −0,07 dan 0,09), sehingga keempat fitur membawa informasi yang berbeda-beda dan tidak ada fitur yang redundan. Pola persebaran gempa juga tidak bisa dipisahkan secara tegas hanya dengan dua fitur, karena kelompok gempa ditentukan bersama oleh lokasi, kedalaman, dan magnitudo. Oleh karena itu, keempat fitur digunakan bersamaan dalam proses clustering.

#### 3. Data Preparation

Proses clustering menggunakan empat fitur numerik:
* `latitude` dan `longitude`: lokasi episenter
* `depth`: kedalaman gempa
* `mag`: kekuatan gempa

Kolom lain seperti `time`, `id`, `place`, dan indikator kualitas pengukuran tidak digunakan karena tidak menggambarkan karakteristik fisik gempa. Kolom tersebut tetap disimpan untuk interpretasi hasil.

In [ ]:
features = ["latitude", "longitude", "depth", "mag"]

# Menghapus baris yang fiturnya kosong (jika ada) dan data duplikat
df = df.dropna(subset=features).drop_duplicates(subset="id").reset_index(drop=True)

X = df[features]
X.shape

Skala keempat fitur sangat berbeda: kedalaman bisa mencapai ratusan km, sementara magnitudo hanya berkisar 4,5–8. Tanpa standardisasi, fitur `depth` akan mendominasi perhitungan jarak. Oleh karena itu, dilakukan standardisasi menggunakan `StandardScaler` sehingga setiap fitur memiliki rata-rata 0 dan standar deviasi 1.

Hasil standardisasi disimpan pada variabel terpisah (`scaled_data`) sehingga nilai asli di `df` tetap utuh untuk interpretasi.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_data = pd.DataFrame(
    scaler.fit_transform(X),
    columns=features
)

scaled_data.head()

In [ ]:
scaled_data.describe().round(2)

#### 4. Modeling

Data yang telah distandardisasi digunakan untuk clustering dengan tiga algoritma: K-Means, Hierarchical Clustering, dan DBSCAN.

##### **4.1. K-Means Clustering**

Jumlah cluster K ditentukan menggunakan Elbow Method. Inertia mengukur total jarak kuadrat data terhadap centroid clusternya; semakin kecil inertia, semakin rapat cluster.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_RANGE = range(2, 11)
inertia = []
silhouette_scores = []

for k in K_RANGE:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(scaled_data, labels))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_RANGE, inertia, marker="o")
axes[0].set_xlabel("Jumlah Cluster (K)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(K_RANGE, silhouette_scores, marker="o", color="tab:orange")
axes[1].set_xlabel("Jumlah Cluster (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score K-Means")

plt.tight_layout()
plt.show()

pd.DataFrame({
    "K": list(K_RANGE),
    "Inertia": inertia,
    "Silhouette": silhouette_scores
}).round(4)

Berdasarkan grafik Elbow, penurunan inertia cukup tajam dari K = 2 hingga K = 5 (26.976 → 13.115), lalu mulai melandai setelah K = 5 (13.115 → 11.345 → 9.996). Titik siku berada di **K = 5**.

Hasil ini diperkuat oleh Silhouette Score, yang mencapai nilai tertinggi juga pada **K = 5 (0,3474)**, sedikit di atas K = 3 (0,3431). Karena Elbow Method dan Silhouette Score menunjuk nilai yang sama, jumlah cluster untuk K-Means ditetapkan sebanyak 5.

In [ ]:
# Memilih K dengan Silhouette tertinggi di rentang 3–8
sil_table = dict(zip(K_RANGE, silhouette_scores))
K_OPTIMAL = max(range(3, 9), key=lambda k: sil_table[k])
print("K yang dipilih:", K_OPTIMAL, "| Silhouette:", round(sil_table[K_OPTIMAL], 4))

In [ ]:
kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(scaled_data)

df["KMeans_Cluster"] = kmeans_labels
df["KMeans_Cluster"].value_counts().sort_index()

In [ ]:
def plot_cluster_map(label_col, title):
    """Scatter peta (longitude vs latitude) dan penampang kedalaman, diwarnai per cluster."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [1.3, 1]})
    labels = df[label_col]
    palette = sns.color_palette("tab10", n_colors=max(labels.max() + 1, 1))

    for ax, ycol in zip(axes, ["latitude", "depth"]):
        # Noise DBSCAN (-1) digambar abu-abu
        noise = labels == -1
        if noise.any():
            ax.scatter(df.loc[noise, "longitude"], df.loc[noise, ycol],
                       c="lightgray", s=6, label="Noise")
        for c in sorted(labels[labels != -1].unique()):
            m = labels == c
            ax.scatter(df.loc[m, "longitude"], df.loc[m, ycol],
                       color=palette[c], s=8, alpha=0.7, label=f"Cluster {c}")
        ax.set_xlabel("Longitude")

    axes[0].set_ylabel("Latitude")
    axes[0].set_title(title + " (Peta Episenter)")
    axes[0].legend(markerscale=3, loc="upper right", fontsize=8)
    axes[1].set_ylabel("Kedalaman (km)")
    axes[1].invert_yaxis()
    axes[1].set_title(title + " (Penampang Kedalaman)")

    plt.tight_layout()
    plt.show()

plot_cluster_map("KMeans_Cluster", "Hasil Clustering K-Means")

K-Means membagi 8.731 gempa ke dalam 5 cluster. Dari peta episenter terlihat bahwa empat cluster terbentuk berdasarkan **wilayah** (Sumatra–Jawa, busur selatan Nusa Tenggara–Banda–Papua, kawasan utara Maluku–Sulawesi Utara) dan **kedalaman** (cluster khusus gempa dalam). Satu cluster lainnya tersebar di seluruh wilayah; cluster ini dibentuk oleh **magnitudo** yang tinggi, bukan oleh lokasi. Detail tiap cluster dibahas pada bagian Interpretasi Cluster.

##### **4.2. Hierarchical Clustering**

Pada Hierarchical Clustering (Agglomerative, metode *ward*), penentuan jumlah cluster dilakukan dengan dendrogram. Karena jumlah data mencapai ribuan, dendrogram ditampilkan dalam mode terpotong (`truncate_mode="lastp"`) yang hanya menampilkan 30 penggabungan terakhir agar tetap terbaca.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(scaled_data, method="ward")

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=90, show_contracted=True)

plt.title("Dendrogram Hierarchical Clustering (30 cluster terakhir)")
plt.xlabel("Kelompok data (angka dalam kurung = jumlah anggota)")
plt.ylabel("Jarak Ward")
plt.show()

Garis vertikal yang panjang pada bagian atas dendrogram menunjukkan penggabungan antar kelompok yang sangat berbeda. Jika dendrogram dipotong pada jarak sekitar 70, garis horizontal tersebut memotong **5 cabang** (oranye, hijau, merah, ungu, dan cokelat), sehingga dendrogram mendukung pembentukan 5 cluster. Jumlah ini sama dengan hasil K-Means, sehingga kedua algoritma dapat dibandingkan secara konsisten.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

hierarchical = AgglomerativeClustering(n_clusters=K_OPTIMAL, linkage="ward")
hierarchical_labels = hierarchical.fit_predict(scaled_data)

df["Hierarchical_Cluster"] = hierarchical_labels
df["Hierarchical_Cluster"].value_counts().sort_index()

In [ ]:
plot_cluster_map("Hierarchical_Cluster", "Hasil Hierarchical Clustering")

##### **4.3. DBSCAN**

DBSCAN tidak membutuhkan jumlah cluster di awal. Algoritma ini mengelompokkan data berdasarkan kepadatan, sehingga cocok untuk data gempa: gempa yang terkonsentrasi di satu zona akan membentuk cluster, sedangkan gempa yang terisolasi otomatis dianggap **noise** (label −1).

Dua parameter utama:
* `eps`: radius maksimum antar data dalam satu neighborhood.
* `min_samples`: jumlah minimum data agar suatu area dianggap padat.

Nilai `eps` awal dapat diperkirakan menggunakan **k-distance plot**: jarak setiap data ke tetangga ke-k diurutkan, lalu dicari titik lekukan (knee).

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 10
nn = NearestNeighbors(n_neighbors=k).fit(scaled_data)
distances, _ = nn.kneighbors(scaled_data)
k_dist = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_dist)
plt.xlabel("Data (diurutkan)")
plt.ylabel(f"Jarak ke tetangga ke-{k}")
plt.title("k-Distance Plot untuk Menentukan eps")
plt.grid(alpha=0.3)
plt.show()

Titik lekukan pada grafik di atas menjadi acuan rentang `eps`. Selanjutnya dicoba beberapa kombinasi `eps` dan `min_samples`, dan untuk setiap kombinasi dicatat jumlah cluster, jumlah noise, persentase noise, dan Silhouette Score (dihitung tanpa noise).

In [ ]:
from sklearn.cluster import DBSCAN

dbscan_results = []

for eps in [0.2, 0.3, 0.4, 0.5, 0.6]:
    for min_samples in [5, 10, 20]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(scaled_data)

        mask = labels != -1
        n_clusters = len(set(labels[mask]))
        n_noise = int((~mask).sum())

        if n_clusters >= 2:
            score = silhouette_score(scaled_data[mask], labels[mask])
        else:
            score = np.nan

        dbscan_results.append({
            "eps": eps,
            "min_samples": min_samples,
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "noise_%": round(100 * n_noise / len(labels), 2),
            "silhouette": score
        })

dbscan_results = pd.DataFrame(dbscan_results)
dbscan_results

Memilih parameter hanya berdasarkan Silhouette Score berisiko, karena Silhouette dihitung tanpa noise. Kombinasi yang membuang banyak data sebagai noise bisa terlihat bagus padahal banyak gempa tidak masuk cluster mana pun. Karena itu, parameter terbaik dipilih dengan kriteria:
* jumlah cluster antara 2 dan 15,
* persentase noise maksimal 10%,
* lalu Silhouette Score tertinggi.

In [ ]:
candidates = dbscan_results[
    (dbscan_results["n_clusters"].between(2, 15)) &
    (dbscan_results["noise_%"] <= 10)
].dropna(subset=["silhouette"])

# Jika tidak ada kombinasi yang memenuhi batas noise, longgarkan batasnya
if candidates.empty:
    candidates = dbscan_results[dbscan_results["n_clusters"].between(2, 15)].dropna(subset=["silhouette"])

best = candidates.sort_values("silhouette", ascending=False).iloc[0]
BEST_EPS, BEST_MIN_SAMPLES = float(best["eps"]), int(best["min_samples"])
best

In [ ]:
dbscan = DBSCAN(eps=BEST_EPS, min_samples=BEST_MIN_SAMPLES)
dbscan_labels = dbscan.fit_predict(scaled_data)

df["DBSCAN_Cluster"] = dbscan_labels
df["DBSCAN_Cluster"].value_counts().sort_index()

In [ ]:
plot_cluster_map("DBSCAN_Cluster", "Hasil Clustering DBSCAN")

Dengan parameter terbaik (`eps` = 0,6 dan `min_samples` = 20), DBSCAN hanya menghasilkan **2 cluster** dan **353 noise (4,04%)**:
* **Cluster 0** berisi 8.349 gempa (95,6% data), yaitu hampir seluruh gempa di Indonesia digabung menjadi satu kelompok besar.
* **Cluster 1** hanya berisi 29 gempa, yaitu kelompok gempa sangat dalam (± 500–600 km) di Laut Sulawesi bagian utara.

Titik abu-abu merupakan noise, yaitu gempa yang terisolasi, sebagian besar gempa dalam di bawah Laut Jawa dan Laut Flores. Hal ini terjadi karena zona-zona gempa di Indonesia saling bersambung sepanjang busur kepulauan, sehingga dari sudut pandang kepadatan, DBSCAN melihatnya sebagai satu kesatuan.

#### 5. Evaluation

Ketiga algoritma dibandingkan menggunakan:
* **Silhouette Score**: rentang −1 sampai 1, semakin tinggi semakin baik (cluster rapat dan terpisah jelas).
* **Davies-Bouldin Index (DBI)**: semakin rendah semakin baik (rasio sebaran dalam cluster terhadap jarak antar cluster).

Untuk DBSCAN, metrik dihitung **tanpa noise**, sehingga persentase data yang menjadi noise juga ditampilkan agar perbandingannya adil.

In [ ]:
from sklearn.metrics import davies_bouldin_score

def evaluate(name, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    if n_clusters >= 2:
        sil = silhouette_score(scaled_data[mask], labels[mask])
        dbi = davies_bouldin_score(scaled_data[mask], labels[mask])
    else:
        sil, dbi = np.nan, np.nan
    return {
        "Algorithm": name,
        "Jumlah Cluster": n_clusters,
        "Noise (%)": round(100 * (~mask).mean(), 2),
        "Silhouette Score": sil,
        "Davies-Bouldin Index": dbi
    }

evaluation = pd.DataFrame([
    evaluate("K-Means", kmeans_labels),
    evaluate("Hierarchical", hierarchical_labels),
    evaluate("DBSCAN", dbscan_labels)
])

evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=evaluation, x="Algorithm", y="Silhouette Score", ax=axes[0])
axes[0].set_title("Silhouette Score (semakin tinggi semakin baik)")

sns.barplot(data=evaluation, x="Algorithm", y="Davies-Bouldin Index", ax=axes[1])
axes[1].set_title("Davies-Bouldin Index (semakin rendah semakin baik)")

plt.tight_layout()
plt.show()

In [ ]:
best_sil = evaluation.loc[evaluation["Silhouette Score"].idxmax(), "Algorithm"]
best_dbi = evaluation.loc[evaluation["Davies-Bouldin Index"].idxmin(), "Algorithm"]
print("Silhouette tertinggi   :", best_sil)
print("Davies-Bouldin terendah:", best_dbi)

**Pembahasan hasil evaluasi:**

| Algoritma | Jumlah Cluster | Noise | Silhouette | DBI |
|---|---|---|---|---|
| K-Means | 5 | 0% | 0,3474 | 0,9747 |
| Hierarchical | 5 | 0% | 0,3151 | 1,0499 |
| DBSCAN | 2 | 4,04% | 0,5615 | 0,4005 |

* Secara angka, **DBSCAN** unggul pada kedua metrik: Silhouette tertinggi (0,5615) dan DBI terendah (0,4005).
* Namun keunggulan tersebut perlu dibaca hati-hati. DBSCAN hanya membentuk 2 cluster yang sangat tidak seimbang (8.349 vs 29 gempa), dan 4,04% data dibuang sebagai noise sebelum metrik dihitung. Skor tinggi muncul karena kelompok kecil gempa sangat dalam tersebut memang terpisah jauh dari yang lain, **bukan** karena DBSCAN berhasil memetakan zona-zona gempa Indonesia. Hasil 1 cluster besar tidak memberikan informasi zonasi yang berguna.
* Di antara algoritma yang memasukkan seluruh data ke dalam cluster, **K-Means lebih baik daripada Hierarchical Clustering** pada kedua metrik (Silhouette 0,3474 vs 0,3151; DBI 0,9747 vs 1,0499).
* Nilai Silhouette K-Means sebesar 0,35 menunjukkan struktur cluster yang **cukup (moderat)**: cluster terbentuk, tetapi masih ada tumpang tindih di daerah perbatasan antar wilayah. Hal ini wajar karena jalur gempa di Indonesia bersambung dan tidak memiliki batas yang tegas.
* Dengan mempertimbangkan metrik dan kegunaan hasil, **K-Means dipilih sebagai algoritma terbaik** untuk tujuan pemetaan zona gempa pada studi ini.

#### 6. Interpretasi Cluster

Untuk memahami arti setiap cluster, dihitung profil rata-rata fitur dalam skala aslinya, kategori kedalaman dominan, dan wilayah yang paling sering muncul pada kolom `place`. Interpretasi di bawah menggunakan hasil **K-Means**; ganti `CLUSTER_COL` untuk melihat algoritma lain.

Kategori kedalaman mengikuti klasifikasi umum seismologi: **dangkal** (< 70 km), **menengah** (70–300 km), dan **dalam** (> 300 km).

In [ ]:
CLUSTER_COL = "KMeans_Cluster"

df["kategori_kedalaman"] = pd.cut(
    df["depth"],
    bins=[-np.inf, 70, 300, np.inf],
    labels=["Dangkal", "Menengah", "Dalam"]
)

# Ambil nama wilayah dari kolom place, contoh: "120 km SW of Bengkulu, Indonesia" -> "Bengkulu, Indonesia"
df["wilayah"] = (
    df["place"].astype(str)
    .str.replace(r"^.*? of ", "", regex=True)
    .str.strip()
)

profile = df[df[CLUSTER_COL] != -1].groupby(CLUSTER_COL).agg(
    jumlah_gempa=("mag", "size"),
    lat_rata2=("latitude", "mean"),
    lon_rata2=("longitude", "mean"),
    depth_rata2=("depth", "mean"),
    mag_rata2=("mag", "mean"),
    mag_maks=("mag", "max"),
    kedalaman_dominan=("kategori_kedalaman", lambda s: s.mode().iat[0]),
    wilayah_dominan=("wilayah", lambda s: ", ".join(s.value_counts().index[:3]))
).round(2)

profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, title in zip(
    axes,
    ["depth", "mag", "longitude"],
    ["Kedalaman (km)", "Magnitudo", "Longitude (barat → timur)"]
):
    sns.boxplot(data=df[df[CLUSTER_COL] != -1], x=CLUSTER_COL, y=col, ax=ax)
    ax.set_title(title)

plt.tight_layout()
plt.show()

**Interpretasi cluster K-Means** (berdasarkan tabel `profile`, boxplot, dan peta cluster):

| Cluster | Jumlah | Nama Cluster | Karakteristik |
|---|---|---|---|
| 0 | 708 | Gempa kuat di seluruh Indonesia | Magnitudo rata-rata 5,65 (tertinggi, maks 7,6), kedalaman rata-rata 46 km. Lokasinya tersebar dari Sumatra hingga Papua; cluster ini terbentuk karena **kekuatan gempa**, bukan lokasi |
| 1 | 2.855 | Busur selatan Nusa Tenggara–Laut Banda–Papua | Longitude rata-rata 127,9, latitude −6,2; magnitudo rata-rata 4,70; kedalaman rata-rata 66 km (dangkal–menengah) |
| 2 | 3.146 | Kawasan utara: Laut Maluku–Halmahera–Sulawesi Utara | Cluster terbesar; latitude rata-rata 2,1 (satu-satunya di utara khatulistiwa), longitude 126,4; magnitudo rata-rata 4,70; kedalaman rata-rata 62 km |
| 3 | 1.695 | Zona subduksi Sumatra–Jawa bagian barat | Longitude rata-rata 101,9 (paling barat), kedalaman rata-rata 43 km (dangkal), magnitudo rata-rata 4,77 |
| 4 | 327 | Gempa dalam Laut Jawa–Flores–Banda | Kedalaman rata-rata 474 km (satu-satunya cluster gempa dalam), longitude rata-rata 122,6; magnitudo rata-rata 4,82 |

Beberapa temuan penting:
* **Cluster 1, 2, dan 3** memisahkan gempa berdasarkan wilayah. Kawasan timur Indonesia (Maluku, Laut Banda, Sulawesi Utara, Papua) menyumbang lebih dari 6.000 gempa (cluster 1 + 2), jauh lebih banyak dibandingkan wilayah barat (cluster 3), sehingga kawasan timur merupakan zona gempa paling aktif.
* **Cluster 4** memisahkan gempa berdasarkan kedalaman. Gempa dalam cenderung kurang merusak di permukaan dibandingkan gempa dangkal dengan magnitudo yang sama.
* **Cluster 0** memisahkan gempa berdasarkan magnitudo. Cluster ini paling penting dari sisi mitigasi karena berisi gempa-gempa yang paling berpotensi merusak.


#### 7. Deployment

Pada tahap ini, artefak hasil pelatihan diekspor menggunakan library **Joblib** agar dapat digunakan kembali pada aplikasi tanpa perlu training ulang. Model yang di-deploy adalah **K-Means**, karena model ini terpilih sebagai algoritma terbaik pada tahap Evaluation dan dapat mengelompokkan data baru menggunakan `predict()`.

> Hierarchical Clustering dan DBSCAN pada scikit-learn tidak memiliki method `predict()` untuk data baru, sehingga tidak praktis untuk di-deploy.

Artefak yang disimpan:
| File | Isi |
|---|---|
| `scaler_gempa.joblib` | `StandardScaler` yang sudah di-fit pada data training |
| `kmeans_gempa.joblib` | Model K-Means (K = 5) yang sudah dilatih |
| `cluster_info.json` | Nama dan deskripsi setiap cluster hasil interpretasi |
| `data_cluster.csv` | Koordinat, kedalaman, magnitudo, dan label cluster seluruh data (untuk peta di aplikasi) |


In [ ]:

import joblib
import json

# 1. Simpan scaler dan model K-Means
joblib.dump(scaler, "scaler_gempa.joblib")
joblib.dump(kmeans, "kmeans_gempa.joblib")

print("Scaler dan model berhasil disimpan!")



Nama dan deskripsi cluster disimpan ke file JSON sesuai hasil interpretasi pada bagian 6, supaya aplikasi dapat menampilkan arti cluster, bukan hanya nomornya.

> **Catatan:** nomor cluster di bawah sesuai hasil K-Means dengan `random_state=42` pada dataset ini. Jika dataset atau parameter berubah, sesuaikan kembali dengan tabel `profile`.


In [ ]:

cluster_info = {
    0: {"nama": "Gempa Kuat",
        "deskripsi": "Gempa dengan magnitudo tinggi (rata-rata 5,65) yang dapat terjadi di seluruh wilayah Indonesia. Kelompok ini paling berpotensi merusak."},
    1: {"nama": "Busur Selatan Nusa Tenggara - Laut Banda - Papua",
        "deskripsi": "Gempa dangkal hingga menengah (rata-rata 66 km) di busur selatan kawasan timur Indonesia."},
    2: {"nama": "Kawasan Utara Laut Maluku - Halmahera - Sulawesi Utara",
        "deskripsi": "Kelompok gempa terbanyak, berada di utara khatulistiwa dengan kedalaman rata-rata 62 km."},
    3: {"nama": "Zona Subduksi Sumatra - Jawa Bagian Barat",
        "deskripsi": "Gempa dangkal (rata-rata 43 km) di jalur subduksi sebelah barat Sumatra hingga selatan Jawa bagian barat."},
    4: {"nama": "Gempa Dalam Laut Jawa - Flores - Banda",
        "deskripsi": "Gempa sangat dalam (rata-rata 474 km) akibat lempeng yang menunjam jauh ke dalam bumi."},
}

# Tambahkan profil rata-rata tiap cluster (dihitung dari data asli)
for c, row in profile.iterrows():
    cluster_info[int(c)].update({
        "jumlah_gempa": int(row["jumlah_gempa"]),
        "depth_rata2": float(row["depth_rata2"]),
        "mag_rata2": float(row["mag_rata2"]),
    })

with open("cluster_info.json", "w") as f:
    json.dump(cluster_info, f, indent=2, ensure_ascii=False)

# Data untuk peta di aplikasi
df[["latitude", "longitude", "depth", "mag", "KMeans_Cluster"]].to_csv("data_cluster.csv", index=False)

print("cluster_info.json dan data_cluster.csv berhasil disimpan!")



Streamlit Community Cloud akan meng-install library sesuai `requirements.txt`. Versi `scikit-learn` harus **sama persis** dengan versi yang dipakai saat menyimpan model, karena file `.joblib` dari versi berbeda bisa gagal dimuat.


In [ ]:
import sklearn
print("scikit-learn:", sklearn.__version__)
print("numpy       :", np.__version__)

requirements = f"""streamlit
pandas
joblib
numpy=={np.__version__}
scikit-learn=={sklearn.__version__}
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print(requirements)


Uji coba: memuat kembali artefak dan memprediksi cluster untuk satu gempa baru.


In [ ]:

scaler_loaded = joblib.load("scaler_gempa.joblib")
kmeans_loaded = joblib.load("kmeans_gempa.joblib")

# Contoh gempa baru: lepas pantai barat Sumatra, kedalaman 25 km, M 5,0
gempa_baru = pd.DataFrame([[-3.5, 100.5, 25.0, 5.0]], columns=features)

gempa_baru_scaled = pd.DataFrame(scaler_loaded.transform(gempa_baru), columns=features)
cluster = int(kmeans_loaded.predict(gempa_baru_scaled)[0])
print(f"Gempa baru masuk ke Cluster {cluster}: {cluster_info[cluster]['nama']}")



#### Deployment Menggunakan Streamlit

Aplikasi Streamlit disimpan pada file `app.py`. Aplikasi ini:
1. Memuat `scaler_gempa.joblib`, `kmeans_gempa.joblib`, dan `cluster_info.json`.
2. Menerima input latitude, longitude, kedalaman, dan magnitudo dari pengguna.
3. Melakukan standardisasi input dengan scaler, lalu memprediksi cluster dengan K-Means.
4. Menampilkan nama dan deskripsi cluster, serta posisi gempa pada peta bersama data historis.
5. Menyediakan fitur upload CSV untuk memprediksi banyak gempa sekaligus.

**Langkah deploy ke Streamlit Community Cloud:**
1. Buat repository GitHub baru, lalu upload file: `app.py`, `requirements.txt`, `scaler_gempa.joblib`, `kmeans_gempa.joblib`, `cluster_info.json`, dan `data_cluster.csv`.
2. Buka https://share.streamlit.io dan login menggunakan akun GitHub.
3. Klik **Create app**, pilih repository tersebut, isi *Main file path* dengan `app.py`, lalu klik **Deploy**.
4. Tunggu proses instalasi selesai, lalu aplikasi dapat diakses melalui link `https://<nama-app>.streamlit.app`.

Untuk menjalankan secara lokal: `streamlit run app.py`


#### Kesimpulan

1. Dataset katalog gempa USGS wilayah Indonesia 2015–2024 (M ≥ 4,5) sebanyak 8.731 data dan 22 kolom berhasil dikelompokkan menggunakan empat fitur numerik: latitude, longitude, depth, dan mag.
2. Jumlah cluster optimal untuk K-Means adalah 5, didukung oleh Elbow Method, Silhouette Score tertinggi (0,3474), dan dendrogram Hierarchical Clustering.
3. Secara metrik, DBSCAN memperoleh Silhouette tertinggi (0,5615) dan DBI terendah (0,4005). Namun DBSCAN hanya membentuk 2 cluster yang sangat tidak seimbang (8.349 vs 29 gempa) dengan 4,04% noise, sehingga tidak berguna untuk pemetaan zona gempa.
4. K-Means dipilih sebagai algoritma terbaik untuk studi ini karena mengungguli Hierarchical Clustering pada kedua metrik (Silhouette 0,3474 vs 0,3151; DBI 0,9747 vs 1,0499) dan menghasilkan cluster yang dapat diinterpretasikan secara geologis.
5. Cluster K-Means mencerminkan struktur tektonik Indonesia: tiga cluster berdasarkan wilayah (Sumatra–Jawa barat, busur selatan Nusa Tenggara–Banda–Papua, dan kawasan utara Maluku–Sulawesi Utara), satu cluster gempa dalam (rata-rata 474 km), dan satu cluster gempa kuat (magnitudo rata-rata 5,65).
6. Kawasan timur Indonesia merupakan zona dengan aktivitas gempa tertinggi, sehingga dapat menjadi prioritas utama dalam perencanaan mitigasi bencana gempa bumi.
7. Model K-Means beserta scaler disimpan menggunakan Joblib dan di-deploy sebagai aplikasi web Streamlit, sehingga pengguna dapat memasukkan data gempa baru dan langsung mengetahui zona/kelompok gempanya.